# KAWACH — Counterfeit Currency CNN Training (Kaggle Notebook)

Trains Classifier Pipeline 7's CNN (`/classify-currency`). Targets the ET PS bullet: *"Computer vision AI ... identifies fake notes through microprint analysis, security thread verification, serial number pattern validation, and UV feature simulation ... across all denominations"* — and the evaluation focus: *"Counterfeit detection accuracy across denominations and print quality ... false positive rate for citizen-facing tools (must be very low)."*

**Why Kaggle instead of Colab:** datasets attach as *mounted inputs* — no downloads, no API token, nothing to silently fail. The previous Colab run trained on only 121 images because the largest dataset (7.7 GB) never downloaded; here it's just... there.

## Setup (do this BEFORE Run All)

1. **Add the datasets** — right panel → **Input** → **+ Add Input** → search each slug and add it:

   | Dataset (search by this name) | Size | What it contributes |
   |---|---|---|
   | `preetrank/indian-currency-real-vs-fake-notes-dataset` | **7.7 GB** | The main real/fake corpus — REQUIRED |
   | `devanandjoly/indian-currency-images-for-fake-currency-detection` | 338 MB | More real/fake volume |
   | `jayaprakashpondy/indian-currency-dataset` | 302 MB | Denomination coverage (genuine notes) |
   | `mdladla/fake-currency-data` | 45 MB | Most-voted fake-currency set |
   | `iayushanand/currency-dataset500-inr-note-real-fake` | 61 MB | ₹500 real/fake |
   | `sreeharisureshkaggle/fake-currency-detection-dataset` | 21 MB | ₹500+₹2000 + security-feature crops |

2. **Enable GPU** — right panel → Session options → Accelerator → **GPU T4 x2** (or P100).
3. **Run All.** With the full 7.7 GB corpus expect a real training run (roughly 1–3 hours, not 2 minutes). If it finishes suspiciously fast again, cell 4's per-source table will show you exactly which dataset under-contributed and why.

## Design decisions
- **EfficientNet-B0** — best accuracy/CPU-latency tradeoff for the free-tier serving host (HF Spaces 2vCPU). Checkpoint format is self-describing and matches `Classifier/app/currency_detector.py` exactly — drop-in, zero code changes.
- **Perceptual-hash dedup** before splitting — merged public datasets overlap; near-duplicates across train/val inflate accuracy dishonestly.
- **Stratified 70/15/15 split, test touched once**, per-denomination metrics — the only accuracy numbers you can defend under the PS's own evaluation focus.
- **Per-class cap** (`MAX_PER_CLASS`) keeps epoch time sane on huge corpora while preserving diversity (random sample, seeded).

In [ ]:
import os, re, json, random, warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| GPUs:", torch.cuda.device_count())
if DEVICE == "cpu":
    print("WARNING: no GPU — enable it in Session options > Accelerator, then restart & run all.")

# ── Config ────────────────────────────────────────────────────────────────────
ARCH = "efficientnet_b0"     # "efficientnet_b0" | "mobilenet_v3_small" | "mobilenet_v3_large"
INPUT_SIZE = 224
BATCH_SIZE = 64
HEAD_EPOCHS = 3
FINETUNE_EPOCHS = 15
HEAD_LR = 1e-3
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
EARLY_STOP_PATIENCE = 4
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
MAX_PER_CLASS = 15000        # cap per class AFTER dedup (random seeded sample) — keeps epochs sane
NUM_WORKERS = 4

INPUT_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working")
CURRENCY_CLASSES = ["fake", "real"]  # keep in sync with Classifier/app/currency_detector.py

## 1. Discover attached datasets

Walks `/kaggle/input` and reports raw image counts per attached dataset. **If `preetrank...` is missing or tiny here, stop and fix the Input attachment before continuing** — it's the main corpus.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_dataset_roots(input_root):
    """
    Kaggle mounts attached datasets in one of two layouts:
      old: /kaggle/input/<dataset-slug>/...
      new: /kaggle/input/datasets/<owner>/<dataset-slug>/...
    Return a list of (display_name, root_path), one per actual dataset —
    never lumping all datasets into one 'datasets' blob.
    """
    roots = []
    for top in sorted(p for p in input_root.iterdir() if p.is_dir()):
        if top.name == "datasets":
            # new layout: descend owner/slug
            for owner in sorted(p for p in top.iterdir() if p.is_dir()):
                for slug in sorted(p for p in owner.iterdir() if p.is_dir()):
                    roots.append((f"{owner.name}/{slug.name}", slug))
        else:
            roots.append((top.name, top))
    return roots

dataset_roots = find_dataset_roots(INPUT_ROOT)
if not dataset_roots:
    raise SystemExit("No datasets attached. Use + Add Input (see setup instructions at the top).")

dataset_info = []
for name, root in dataset_roots:
    imgs = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    dataset_info.append({"mount": root, "name": name, "n_images": len(imgs)})
    note = "" if imgs else "   <- no images (CSV/feature-only dataset, will be skipped)"
    print(f"[{name}] {len(imgs)} image files{note}")

total_raw = sum(d["n_images"] for d in dataset_info)
print(f"\nTotal raw images across {len(dataset_info)} dataset(s): {total_raw}")
if len(dataset_info) < 5:
    print("NOTE: expected ~6 datasets. If fewer appear above, check the Input panel attachments.")
if total_raw < 2000:
    print("WARNING: fewer than 2,000 raw images — training on a tiny corpus produces a useless model.")

## 2. Consolidate into one labeled manifest

Label heuristics + per-dataset forced labels, with a **per-source × per-label table** at the end — the diagnostic that catches a dataset silently contributing nothing.

In [ ]:
REAL_KEYWORDS = ["real", "genuine", "original", "authentic", "true", "orig", "actual", "legit", "valid"]
FAKE_KEYWORDS = ["fake", "counterfeit", "forged", "forge", "duplicate", "false", "spurious", "fraud", "bogus"]
DENOM_RE = re.compile(r"(?<!\d)(2000|500|200|100|50|20|10)(?!\d)")

# Datasets that are entirely genuine notes (denomination-recognition sets) —
# matched by substring against the dataset display name (owner/slug).
ALL_REAL_DATASETS = ["jayaprakashpondy"]  # 'Indian Currency Dataset' = denominations, all genuine

# path-substring -> forced label, for inner folder conventions the keywords
# miss (numeric class dirs etc). Check the unclassified sample below.
MANUAL_OVERRIDES = {}

def classify_label(rel_path, dataset_name):
    if any(tag in dataset_name.lower() for tag in ALL_REAL_DATASETS):
        return "real"
    p = rel_path.lower()
    for substr, label in MANUAL_OVERRIDES.items():
        if substr.lower() in p:
            return label
    has_real = any(k in p for k in REAL_KEYWORDS)
    has_fake = any(k in p for k in FAKE_KEYWORDS)
    if has_fake and not has_real:
        return "fake"
    if has_real and not has_fake:
        return "real"
    return None

def classify_denomination(path_str):
    m = DENOM_RE.search(path_str)
    return m.group(1) if m else "unknown"

rows, unclassified = [], []
for d in dataset_info:
    mount, name = d["mount"], d["name"]
    for fp in mount.rglob("*"):
        if fp.is_file() and fp.suffix.lower() in IMG_EXTS:
            # CRITICAL: classify on the path INSIDE the dataset only. The
            # dataset slugs themselves contain both 'real' and 'fake'
            # (e.g. indian-currency-real-vs-fake-notes-dataset), which made
            # every file ambiguous when full paths were used — that's what
            # silently discarded ~12.5k images in earlier runs.
            rel = str(fp.relative_to(mount))
            label = classify_label(rel, name)
            if label is None:
                unclassified.append((name, rel))
                continue
            rows.append({"path": str(fp), "label": label,
                         "denomination": classify_denomination(rel), "source": name})

manifest = pd.DataFrame(rows)
print(f"Classified: {len(manifest)} | Unclassified (skipped): {len(unclassified)}")

if unclassified:
    unc_by_source = Counter(s for s, _ in unclassified)
    print("\nUnclassified by source (add MANUAL_OVERRIDES if these are usable data):")
    for s, n in unc_by_source.most_common():
        print(f"  {s}: {n}")
    print("Sample unclassified relative paths:")
    for s, p in unclassified[:10]:
        print(f"  [{s}] {p}")

print("\nPer-source contribution (source x label) — every image dataset should have nonzero rows:")
print(manifest.groupby(["source", "label"]).size().unstack(fill_value=0))
print("\nLabel distribution:\n", manifest["label"].value_counts())
print("\nDenomination distribution:\n", manifest["denomination"].value_counts())

## 3. Deduplicate (perceptual hash, parallel)

Thumbnails + threaded hashing so this stays fast even on tens of thousands of images.

In [ ]:
import imagehash
from PIL import Image

def phash_of(path):
    try:
        with Image.open(path) as im:
            im.thumbnail((256, 256))
            return str(imagehash.phash(im.convert("RGB")))
    except Exception:
        return None

with ThreadPoolExecutor(max_workers=8) as ex:
    manifest["phash"] = list(ex.map(phash_of, manifest["path"].tolist()))

before_by_source = manifest["source"].value_counts()
before = len(manifest)
manifest = manifest.dropna(subset=["phash"]).drop_duplicates(subset=["phash"], keep="first")
after_by_source = manifest["source"].value_counts()

print(f"Deduplicated: {before} -> {len(manifest)} ({before - len(manifest)} near-duplicates removed)")
print("\nPer-source before -> after:")
for src in before_by_source.index:
    print(f"  {src}: {before_by_source.get(src, 0)} -> {after_by_source.get(src, 0)}")
print("\nPost-dedup label distribution:\n", manifest["label"].value_counts())

## 4. Cap per class + stratified train/val/test split

In [ ]:
# Cap AFTER dedup: random seeded sample per class keeps diversity, bounds epoch time
capped = []
for label, group in manifest.groupby("label"):
    orig_n = len(group)
    if orig_n > MAX_PER_CLASS:
        group = group.sample(MAX_PER_CLASS, random_state=SEED)
        print(f"Capped '{label}' from {orig_n} to {MAX_PER_CLASS}")
    capped.append(group)
manifest = pd.concat(capped).reset_index(drop=True)

manifest["strata"] = manifest["label"] + "_" + manifest["denomination"]

def safe_stratify(df, min_members=2):
    """Return a stratify column that's guaranteed splittable for THIS frame:
    strata with too few members in this specific frame fall back to label;
    if even a label is a singleton, fall back to no stratification."""
    counts = df["strata"].value_counts()
    col = df["strata"].where(df["strata"].map(counts) >= min_members, df["label"])
    if col.value_counts().min() < min_members:
        return None  # unstratified split as last resort
    return col

train_df, temp_df = train_test_split(
    manifest, test_size=(VAL_FRAC + TEST_FRAC),
    stratify=safe_stratify(manifest), random_state=SEED)

val_df, test_df = train_test_split(
    temp_df, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
    stratify=safe_stratify(temp_df), random_state=SEED)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name}: {dict(df['label'].value_counts())}")
if len(train_df) < 500:
    print("\nWARNING: under 500 training images — accuracy from this run is NOT defensible. "
          "Fix the data problem (see per-source table in section 2) before trusting results.")

## 5. Augmentation + loaders

Domain-realistic: perspective warp, motion blur, phone JPEG artifacts, lighting jitter — simulating how citizens/tellers actually photograph notes. No MixUp/CutMix (blending real+fake pixels fights the crisp security-feature cues the model must learn).

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = A.Compose([
    A.LongestMaxSize(max_size=int(INPUT_SIZE * 1.15)),
    A.PadIfNeeded(min_height=int(INPUT_SIZE * 1.15), min_width=int(INPUT_SIZE * 1.15),
                  border_mode=cv2.BORDER_CONSTANT, value=(255, 255, 255)),
    A.RandomCrop(INPUT_SIZE, INPUT_SIZE),
    A.Perspective(scale=(0.02, 0.08), p=0.5),
    A.Rotate(limit=12, border_mode=cv2.BORDER_CONSTANT, value=(255, 255, 255), p=0.6),
    A.OneOf([A.MotionBlur(blur_limit=5), A.GaussianBlur(blur_limit=3)], p=0.35),
    A.ImageCompression(quality_lower=55, quality_upper=95, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
    A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20, val_shift_limit=15, p=0.4),
    A.ISONoise(p=0.25),
    A.CoarseDropout(max_holes=3, max_height=int(INPUT_SIZE*0.08), max_width=int(INPUT_SIZE*0.08), p=0.25),
    A.HorizontalFlip(p=0.3),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.LongestMaxSize(max_size=INPUT_SIZE),
    A.PadIfNeeded(min_height=INPUT_SIZE, min_width=INPUT_SIZE,
                  border_mode=cv2.BORDER_CONSTANT, value=(255, 255, 255)),
    A.CenterCrop(INPUT_SIZE, INPUT_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

class CurrencyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["path"])
        if img is None:
            img = np.zeros((INPUT_SIZE, INPUT_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        out = self.transform(image=img)["image"]
        return out, CURRENCY_CLASSES.index(row["label"])

train_ds = CurrencyDataset(train_df, train_transform)
val_ds = CurrencyDataset(val_df, eval_transform)
test_ds = CurrencyDataset(test_df, eval_transform)

class_counts = train_df["label"].value_counts().to_dict()
sample_weights = train_df["label"].map(lambda l: 1.0 / class_counts[l]).values
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_df), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, drop_last=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, persistent_workers=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"Loaders ready — train batches/epoch: {len(train_loader)}")

## 6. Model — mirrors `Classifier/app/currency_detector.py`'s registry exactly

If you change one side, change the other — the checkpoint must load on the serving side with zero code changes.

In [ ]:
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
    mobilenet_v3_large, MobileNet_V3_Large_Weights,
)

def _swap_efficientnet(model, n_classes):
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, n_classes)
    return model

def _swap_mobilenet(model, n_classes):
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, n_classes)
    return model

ARCH_REGISTRY = {
    "efficientnet_b0": (efficientnet_b0, EfficientNet_B0_Weights.IMAGENET1K_V1, _swap_efficientnet),
    "mobilenet_v3_small": (mobilenet_v3_small, MobileNet_V3_Small_Weights.IMAGENET1K_V1, _swap_mobilenet),
    "mobilenet_v3_large": (mobilenet_v3_large, MobileNet_V3_Large_Weights.IMAGENET1K_V1, _swap_mobilenet),
}

def build_currency_model(arch=ARCH, pretrained=False):
    ctor, weights_enum, swap_fn = ARCH_REGISTRY[arch]
    model = ctor(weights=weights_enum if pretrained else None)
    return swap_fn(model, len(CURRENCY_CLASSES))

def set_backbone_trainable(model, trainable):
    for name, p in model.named_parameters():
        if "classifier" not in name:
            p.requires_grad = trainable

model = build_currency_model(ARCH, pretrained=True).to(DEVICE)
print(f"Arch: {ARCH} | Params: {sum(p.numel() for p in model.parameters()):,}")

## 7. Train — Phase 1 (head warmup) then Phase 2 (full fine-tune)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total, correct, loss_sum = 0, 0, 0.0
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        if train_mode:
            optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE, enabled=(DEVICE == "cuda")):
            out = model(x)
            loss = criterion(out, y)
        if train_mode:
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            scaler.step(optimizer)
            scaler.update()
        loss_sum += loss.item() * len(y)
        preds = out.argmax(1)
        correct += (preds == y).sum().item()
        total += len(y)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y.cpu().tolist())
    return loss_sum / total, correct / total, f1_score(all_labels, all_preds, average="macro", zero_division=0)

def train_loop(model, epochs, lr, phase_name):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                   lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    best_f1, best_state, patience_left = -1.0, None, EARLY_STOP_PATIENCE
    history = []
    for epoch in range(epochs):
        tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, criterion, optimizer, scaler)
        val_loss, val_acc, val_f1 = run_epoch(model, val_loader, criterion)
        scheduler.step()
        history.append({"phase": phase_name, "epoch": epoch + 1,
                        "train_loss": tr_loss, "train_acc": tr_acc, "train_f1": tr_f1,
                        "val_loss": val_loss, "val_acc": val_acc, "val_f1": val_f1})
        print(f"[{phase_name}] {epoch+1}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
              f"val loss {val_loss:.4f} acc {val_acc:.3f} f1 {val_f1:.3f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_left = EARLY_STOP_PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"Early stop at epoch {epoch+1}")
                break
    model.load_state_dict(best_state)
    return model, history, best_f1

set_backbone_trainable(model, False)
model, history_head, _ = train_loop(model, HEAD_EPOCHS, HEAD_LR, "head-warmup")

set_backbone_trainable(model, True)
model, history_ft, best_val_f1 = train_loop(model, FINETUNE_EPOCHS, FINETUNE_LR, "fine-tune")
print(f"\nBest validation macro-F1: {best_val_f1:.4f}")

full_history = pd.DataFrame(history_head + history_ft)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric in zip(axes, ["loss", "acc", "f1"]):
    ax.plot(full_history[f"train_{metric}"].values, label="train")
    ax.plot(full_history[f"val_{metric}"].values, label="val")
    ax.axvline(len(history_head) - 0.5, color="gray", linestyle="--", alpha=0.5)
    ax.set_title(metric); ax.legend()
plt.tight_layout(); plt.show()

## 8. Final evaluation — held-out test set (touched once)

Per-denomination accuracy/precision/recall/F1/AUC + confusion matrix + ROC + threshold sweep. The threshold sweep matters for the PS's "false positive rate must be very low" criterion — pick the operating point from this table, not blindly 0.5.

In [ ]:
model.eval()
all_probs = []
with torch.no_grad():
    for x, y in test_loader:
        probs = torch.softmax(model(x.to(DEVICE)), dim=1)[:, CURRENCY_CLASSES.index("fake")].cpu().numpy()
        all_probs.extend(probs.tolist())

test_df = test_df.reset_index(drop=True)
test_df["fake_prob"] = all_probs
test_df["pred_label"] = ["fake" if p >= 0.5 else "real" for p in all_probs]
test_df["true_idx"] = [CURRENCY_CLASSES.index(l) for l in test_df["label"]]

def compute_metrics(sub_df):
    if len(sub_df) == 0:
        return None
    y_true = sub_df["true_idx"].values
    y_pred = [CURRENCY_CLASSES.index(l) for l in sub_df["pred_label"]]
    fake_idx = CURRENCY_CLASSES.index("fake")
    result = {
        "n": len(sub_df),
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "fake_precision": round(precision_score(y_true, y_pred, pos_label=fake_idx, zero_division=0), 4),
        "fake_recall": round(recall_score(y_true, y_pred, pos_label=fake_idx, zero_division=0), 4),
        "fake_f1": round(f1_score(y_true, y_pred, pos_label=fake_idx, zero_division=0), 4),
    }
    if len(set(y_true)) > 1:
        # AUC with "fake" as the positive class, scored by fake probability.
        # (fake is index 0 in CURRENCY_CLASSES — passing raw true_idx would
        # treat "real" as positive and print the AUC inverted.)
        y_fake = (y_true == fake_idx).astype(int)
        result["auc"] = round(roc_auc_score(y_fake, sub_df["fake_prob"]), 4)
    return result

overall = compute_metrics(test_df)
print("OVERALL:", json.dumps(overall, indent=2))
per_denom = {d: m for d, sub in test_df.groupby("denomination") if (m := compute_metrics(sub))}
print("\nPER-DENOMINATION:")
print(json.dumps(per_denom, indent=2))

cm = confusion_matrix(test_df["true_idx"], [CURRENCY_CLASSES.index(l) for l in test_df["pred_label"]])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CURRENCY_CLASSES, yticklabels=CURRENCY_CLASSES, cmap="Blues", ax=axes[0])
axes[0].set_title("Confusion Matrix (test)")
fake_idx = CURRENCY_CLASSES.index("fake")
y_fake_all = (test_df["true_idx"].values == fake_idx).astype(int)
if len(set(y_fake_all)) > 1:
    fpr, tpr, _ = roc_curve(y_fake_all, test_df["fake_prob"])
    axes[1].plot(fpr, tpr, label=f"AUC = {overall.get('auc', 0):.3f}")
    axes[1].plot([0, 1], [0, 1], "--", color="gray"); axes[1].legend(); axes[1].set_title("ROC (fake = positive)")
plt.tight_layout(); plt.show()

print("\nThreshold sweep (fake_prob >= t => predict fake) — pick the row with the "
      "false-positive tradeoff you want (citizen tool: high precision; teller tool: high recall):")
print(f"{'t':>5} {'accuracy':>9} {'fake_prec':>10} {'fake_recall':>12} {'fake_f1':>8}")
for t in np.arange(0.3, 0.75, 0.05):
    preds = (test_df["fake_prob"] >= t).astype(int)
    y_true = test_df["true_idx"].values
    print(f"{t:>5.2f} {accuracy_score(y_true, preds):>9.3f} "
          f"{precision_score(y_true, preds, pos_label=fake_idx, zero_division=0):>10.3f} "
          f"{recall_score(y_true, preds, pos_label=fake_idx, zero_division=0):>12.3f} "
          f"{f1_score(y_true, preds, pos_label=fake_idx, zero_division=0):>8.3f}")

## 9. Export

Saves to `/kaggle/working` — download from the **Output** panel on the right after the run (or Save Version → Output tab). Copy both files to `Classifier/weights/currency/` in the repo; the service auto-loads on restart (`/health` → `currency_mode: "cnn+heuristic"`).

In [ ]:
checkpoint_path = OUT_DIR / "currency_cnn.pt"
torch.save({
    "arch": ARCH,
    "classes": CURRENCY_CLASSES,
    "state_dict": model.state_dict(),
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "trainer": "kaggle_train_currency.ipynb",
    "test_metrics_summary": overall,
}, checkpoint_path)

report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "arch": ARCH,
    "train_samples": len(train_df),
    "val_samples": len(val_df),
    "test_samples": len(test_df),
    "best_val_macro_f1": round(best_val_f1, 4),
    "overall": overall,
    "per_denomination": per_denom,
    "datasets_used": [d["name"] for d in dataset_info],
    "raw_images_per_dataset": {d["name"]: d["n_images"] for d in dataset_info},
    "note": "Quote per-denomination numbers, never the blended overall figure alone. "
            "If train_samples is small, treat accuracy as provisional (see section 2/4 warnings).",
}
report_path = OUT_DIR / "eval_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Saved: {checkpoint_path} ({checkpoint_path.stat().st_size / 1e6:.1f} MB)")
print(f"Saved: {report_path}")
print(json.dumps(report, indent=2))
print("\nDownload both from the Output panel (right sidebar), then place in Classifier/weights/currency/")